<a href="https://colab.research.google.com/github/vishwaShetti1/New/blob/main/data_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
# ============================================================
# DATA PIPELINE PROJECT
# PART 1 - SCRAPE BOOK DATA FROM books.toscrape.com
# ============================================================

# ------------------------------------------------------------
# Import Required Libraries
# ------------------------------------------------------------

import requests                  # Used to send HTTP requests
from bs4 import BeautifulSoup    # Used to extract HTML data
import pandas as pd              # Used to store data in DataFrame

# ------------------------------------------------------------
# Website URL
# ------------------------------------------------------------

BASE_URL = "https://books.toscrape.com/catalogue/page-{}.html"

# ------------------------------------------------------------
# Create Empty List
# ------------------------------------------------------------

# Every scraped book will be stored in this list
books = []

# ------------------------------------------------------------
# Rating Conversion Dictionary
# ------------------------------------------------------------

# The website stores ratings as words.
# We will later convert them into numbers.

rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

# ------------------------------------------------------------
# Scrape First 5 Pages
# ------------------------------------------------------------

# Each page contains 20 books.
# Therefore, 5 pages = 100 books.

for page in range(1, 6):

    print(f"\nScraping Page {page}...")

    # Create page URL
    url = BASE_URL.format(page)

    # Send request
    response = requests.get(url)

    # Check whether request is successful
    if response.status_code != 200:
        print("Unable to access page:", page)
        continue

    # Convert HTML into BeautifulSoup object
    soup = BeautifulSoup(response.text, "html.parser")

    # Find all books available on the page
    all_books = soup.find_all("article", class_="product_pod")

    # --------------------------------------------------------
    # Loop through every book
    # --------------------------------------------------------

    for book in all_books:

        # ------------------------------
        # Book Title
        # ------------------------------

        title = book.h3.a["title"]

        # ------------------------------
        # Book Price
        # ------------------------------

        price = book.find(
            "p",
            class_="price_color"
        ).text.strip()

        # ------------------------------
        # Book Rating
        # ------------------------------

        rating_text = book.find(
            "p",
            class_="star-rating"
        )["class"][1]

        # ------------------------------
        # Availability
        # ------------------------------

        availability = book.find(
            "p",
            class_="instock availability"
        ).get_text(strip=True)

        # ----------------------------------------------------
        # Category
        # ----------------------------------------------------
        # Open the product page to get the category

        book_link = book.h3.a["href"]

        # Convert relative URL into complete URL
        book_url = (
            "https://books.toscrape.com/catalogue/"
            + book_link.replace("../", "")
        )

        # Open product page
        book_response = requests.get(book_url)

        if book_response.status_code == 200:

            book_soup = BeautifulSoup(
                book_response.text,
                "html.parser"
            )

            breadcrumb = book_soup.find(
                "ul",
                class_="breadcrumb"
            )

            category = breadcrumb.find_all("li")[2].text.strip()

        else:

            category = "Unknown"

        # ----------------------------------------------------
        # Save Book Information
        # ----------------------------------------------------

        books.append({

            "title": title,

            "price": price,

            "star_rating": rating_text,

            "availability": availability,

            "category": category

        })

# ------------------------------------------------------------
# Convert List into DataFrame
# ------------------------------------------------------------

df = pd.DataFrame(books)

# ------------------------------------------------------------
# Display Dataset Information
# ------------------------------------------------------------

print("\n===================================")
print("SCRAPING COMPLETED")
print("===================================")

print("\nTotal Books Scraped :", len(df))

print("\nColumns")
print(df.columns)

print("\nFirst Five Records")
print(df.head())

# ------------------------------------------------------------
# Save Raw Dataset
# ------------------------------------------------------------

df.to_csv("raw_books.csv", index=False)

print("\nRaw dataset saved as raw_books.csv")


Scraping Page 1...

Scraping Page 2...

Scraping Page 3...

Scraping Page 4...

Scraping Page 5...

SCRAPING COMPLETED

Total Books Scraped : 100

Columns
Index(['title', 'price', 'star_rating', 'availability', 'category'], dtype='object')

First Five Records
                                   title    price star_rating availability  \
0                   A Light in the Attic  Â£51.77       Three     In stock   
1                     Tipping the Velvet  Â£53.74         One     In stock   
2                             Soumission  Â£50.10         One     In stock   
3                          Sharp Objects  Â£47.82        Four     In stock   
4  Sapiens: A Brief History of Humankind  Â£54.23        Five     In stock   

             category  
0              Poetry  
1  Historical Fiction  
2             Fiction  
3             Mystery  
4             History  

Raw dataset saved as raw_books.csv


In [14]:
# ============================================================
# PART 2 - DATA CLEANING AND TRANSFORMATION
# ============================================================

import pandas as pd

print("="*60)
print("DATASET INFORMATION BEFORE CLEANING")
print("="*60)

print("\nFirst Five Records")
print(df.head())

print("\nDataset Shape")
print(df.shape)

print("\nColumn Names")
print(df.columns.tolist())

print("\nMissing Values")
print(df.isnull().sum())

# ------------------------------------------------------------
# Clean Price Column
# ------------------------------------------------------------

print("\nCleaning Price Column...")

# Remove encoding characters and currency symbol
df["price"] = (
    df["price"]
    .astype(str)
    .str.replace("Â", "", regex=False)
    .str.replace("£", "", regex=False)
    .str.strip()
)

# Convert to float
df["price_gbp"] = pd.to_numeric(df["price"], errors="coerce")

# Fill missing prices with median
df["price_gbp"] = df["price_gbp"].fillna(df["price_gbp"].median())

# ------------------------------------------------------------
# Convert Ratings
# ------------------------------------------------------------

print("Converting Ratings...")

rating_map = {
    "One":1,
    "Two":2,
    "Three":3,
    "Four":4,
    "Five":5
}

df["rating"] = df["star_rating"].map(rating_map)

# ------------------------------------------------------------
# Convert Availability
# ------------------------------------------------------------

print("Converting Availability...")

df["in_stock"] = df["availability"].str.contains(
    "In stock",
    case=False,
    na=False
)

# ------------------------------------------------------------
# Handle Missing Values
# ------------------------------------------------------------

print("Handling Missing Values...")

df["title"] = df["title"].fillna("Unknown")
df["category"] = df["category"].fillna("Unknown")
df["rating"] = df["rating"].fillna(0).astype(int)

# ------------------------------------------------------------
# Convert GBP to INR
# ------------------------------------------------------------

print("Converting GBP to INR...")

GBP_TO_INR = 105.50

df["price_inr"] = (df["price_gbp"] * GBP_TO_INR).round(2)

# ------------------------------------------------------------
# Check Data Types
# ------------------------------------------------------------

print("\n" + "="*60)
print("DATA TYPES")
print("="*60)

print(df.dtypes)

# ------------------------------------------------------------
# Missing Values After Cleaning
# ------------------------------------------------------------

print("\n" + "="*60)
print("MISSING VALUES AFTER CLEANING")
print("="*60)

print(df.isnull().sum())

# ------------------------------------------------------------
# Display Clean Data
# ------------------------------------------------------------

print("\n" + "="*60)
print("FIRST FIVE CLEANED RECORDS")
print("="*60)

print(df.head())

# ------------------------------------------------------------
# Save Clean Dataset
# ------------------------------------------------------------

df.to_csv("clean_books.csv", index=False)

print("\nCleaned dataset saved as clean_books.csv")

# ------------------------------------------------------------
# Final Summary
# ------------------------------------------------------------

print("\n" + "="*60)
print("DATA CLEANING COMPLETED SUCCESSFULLY")
print("="*60)

print("\nTotal Books :", len(df))

print("\nColumns Available")
print(df.columns.tolist())

DATASET INFORMATION BEFORE CLEANING

First Five Records
                                   title    price star_rating availability  \
0                   A Light in the Attic  Â£51.77       Three     In stock   
1                     Tipping the Velvet  Â£53.74         One     In stock   
2                             Soumission  Â£50.10         One     In stock   
3                          Sharp Objects  Â£47.82        Four     In stock   
4  Sapiens: A Brief History of Humankind  Â£54.23        Five     In stock   

             category  
0              Poetry  
1  Historical Fiction  
2             Fiction  
3             Mystery  
4             History  

Dataset Shape
(100, 5)

Column Names
['title', 'price', 'star_rating', 'availability', 'category']

Missing Values
title           0
price           0
star_rating     0
availability    0
category        0
dtype: int64

Cleaning Price Column...
Converting Ratings...
Converting Availability...
Handling Missing Values...
Converting

In [15]:
# ============================================================
# DATA PIPELINE PROJECT
# PART 3 - CREATE SQLITE DATABASE
# ============================================================

import os
import sqlite3
import pandas as pd

# ------------------------------------------------------------
# Delete old database
# ------------------------------------------------------------

if os.path.exists("books.db"):
    os.remove("books.db")

# ------------------------------------------------------------
# Create SQLite Database
# ------------------------------------------------------------

print("=" * 60)
print("CREATING SQLITE DATABASE")
print("=" * 60)

connection = sqlite3.connect("books.db")
cursor = connection.cursor()

print("Database Connected Successfully")

# ------------------------------------------------------------
# Create Categories Table
# ------------------------------------------------------------

cursor.execute("""
CREATE TABLE categories(
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT UNIQUE
)
""")

print("Categories Table Created")

# ------------------------------------------------------------
# Create Books Table
# ------------------------------------------------------------

cursor.execute("""
CREATE TABLE books(
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    price_gbp REAL,
    price_inr REAL,
    rating INTEGER,
    in_stock INTEGER,
    category_id INTEGER,
    FOREIGN KEY(category_id)
    REFERENCES categories(category_id)
)
""")

connection.commit()

print("Books Table Created")

# ------------------------------------------------------------
# Extract Categories
# ------------------------------------------------------------

print("\nExtracting Categories...")

unique_categories = sorted(df["category"].unique())

for category in unique_categories:
    cursor.execute(
        "INSERT INTO categories(category_name) VALUES(?)",
        (category,)
    )

connection.commit()

print("Categories Inserted Successfully")

# ------------------------------------------------------------
# Read Categories
# ------------------------------------------------------------

category_df = pd.read_sql(
    "SELECT * FROM categories",
    connection
)

print(category_df)

# ------------------------------------------------------------
# Merge Category IDs
# ------------------------------------------------------------

books_df = pd.merge(
    df,
    category_df,
    left_on="category",
    right_on="category_name",
    how="left"
)

print("\nMerge Completed")

print(books_df[[
    "title",
    "price_gbp",
    "price_inr",
    "rating",
    "in_stock",
    "category_id"
]].head())

# ------------------------------------------------------------
# Insert Books
# ------------------------------------------------------------

print("\nInserting Books...")

for _, row in books_df.iterrows():

    cursor.execute(
        """
        INSERT INTO books
        (
            title,
            price_gbp,
            price_inr,
            rating,
            in_stock,
            category_id
        )
        VALUES (?,?,?,?,?,?)
        """,
        (
            row["title"],
            float(row["price_gbp"]),
            float(row["price_inr"]),
            int(row["rating"]),
            int(row["in_stock"]),
            int(row["category_id"])
        )
    )

connection.commit()

print("Books Inserted Successfully")

# ------------------------------------------------------------
# Verify Books
# ------------------------------------------------------------

books = pd.read_sql("""
SELECT
book_id,
title,
price_gbp,
price_inr,
rating,
in_stock,
category_id
FROM books
LIMIT 10
""", connection)

print("\nFirst 10 Books")
print(books)

# ------------------------------------------------------------
# Count Books
# ------------------------------------------------------------

count = pd.read_sql(
    "SELECT COUNT(*) AS Total_Books FROM books",
    connection
)

print("\nTotal Books")
print(count)

# ------------------------------------------------------------
# Count Categories
# ------------------------------------------------------------

cat = pd.read_sql(
    "SELECT COUNT(*) AS Total_Categories FROM categories",
    connection
)

print("\nTotal Categories")
print(cat)

# ------------------------------------------------------------
# Database Tables
# ------------------------------------------------------------

tables = pd.read_sql(
    """
    SELECT name
    FROM sqlite_master
    WHERE type='table'
    """,
    connection
)

print("\nDatabase Tables")
print(tables)

print("\n" + "=" * 60)
print("DATABASE CREATED SUCCESSFULLY")
print("=" * 60)



CREATING SQLITE DATABASE
Database Connected Successfully
Categories Table Created
Books Table Created

Extracting Categories...
Categories Inserted Successfully
    category_id       category_name
0             1       Add a comment
1             2                 Art
2             3            Business
3             4           Childrens
4             5        Contemporary
5             6             Default
6             7             Fantasy
7             8             Fiction
8             9      Food and Drink
9            10              Health
10           11  Historical Fiction
11           12             History
12           13              Horror
13           14               Music
14           15             Mystery
15           16           New Adult
16           17          Nonfiction
17           18          Philosophy
18           19              Poetry
19           20            Politics
20           21             Romance
21           22             Science
22         

In [16]:
# ============================================================
# DATA PIPELINE PROJECT
# PART 4 - SQL QUERIES
# ============================================================

import pandas as pd

print("=" * 70)
print("EXECUTING SQL QUERIES")
print("=" * 70)

# ------------------------------------------------------------
# QUERY 1 : SELECT + WHERE
# ------------------------------------------------------------

print("\nQUERY 1 : SELECT + WHERE")

query1 = """
SELECT
    title,
    rating,
    price_gbp
FROM books
WHERE rating = 5;
"""

result1 = pd.read_sql(query1, connection)

print(result1)

# ------------------------------------------------------------
# QUERY 2 : ORDER BY + LIMIT
# ------------------------------------------------------------

print("\nQUERY 2 : ORDER BY + LIMIT")

query2 = """
SELECT
    title,
    price_gbp,
    price_inr
FROM books
ORDER BY price_gbp DESC
LIMIT 10;
"""

result2 = pd.read_sql(query2, connection)

print(result2)

# ------------------------------------------------------------
# QUERY 3 : DISTINCT
# ------------------------------------------------------------

print("\nQUERY 3 : DISTINCT")

query3 = """
SELECT DISTINCT
    rating
FROM books
ORDER BY rating;
"""

result3 = pd.read_sql(query3, connection)

print(result3)

# ------------------------------------------------------------
# QUERY 4 : BETWEEN
# ------------------------------------------------------------

print("\nQUERY 4 : BETWEEN")

query4 = """
SELECT
    title,
    price_gbp,
    rating
FROM books
WHERE price_gbp BETWEEN 20 AND 40;
"""

result4 = pd.read_sql(query4, connection)

print(result4)

# ------------------------------------------------------------
# QUERY 5 : INNER JOIN
# ------------------------------------------------------------

print("\nQUERY 5 : INNER JOIN")

query5 = """
SELECT
    b.title,
    c.category_name,
    b.rating,
    b.price_gbp
FROM books b
INNER JOIN categories c
ON b.category_id = c.category_id
ORDER BY c.category_name, b.rating DESC;
"""

result5 = pd.read_sql(query5, connection)

print(result5)

# ------------------------------------------------------------
# QUERY 6 : GROUP BY
# ------------------------------------------------------------

print("\nQUERY 6 : GROUP BY")

query6 = """
SELECT
    c.category_name,
    COUNT(*) AS Total_Books,
    ROUND(AVG(b.price_gbp),2) AS Average_Price
FROM books b
INNER JOIN categories c
ON b.category_id = c.category_id
GROUP BY c.category_name
ORDER BY Total_Books DESC;
"""

result6 = pd.read_sql(query6, connection)

print(result6)

# ------------------------------------------------------------
# SAVE OUTPUTS
# ------------------------------------------------------------

print("\nSaving Query Outputs...")

with open("sql_query_outputs.txt", "w", encoding="utf-8") as file:

    file.write("=" * 70 + "\n")
    file.write("SQL QUERY OUTPUTS\n")
    file.write("=" * 70 + "\n\n")

    queries = [
        ("QUERY 1", query1, result1),
        ("QUERY 2", query2, result2),
        ("QUERY 3", query3, result3),
        ("QUERY 4", query4, result4),
        ("QUERY 5", query5, result5),
        ("QUERY 6", query6, result6),
    ]

    for title, sql, result in queries:
        file.write(title + "\n")
        file.write("-" * 70 + "\n")
        file.write(sql.strip())
        file.write("\n\n")
        file.write(result.to_string(index=False))
        file.write("\n\n")

print("Query results saved as sql_query_outputs.txt")

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ALL SQL QUERIES EXECUTED SUCCESSFULLY")
print("=" * 70)

print("\nQueries Covered")
print("✓ SELECT")
print("✓ WHERE")
print("✓ ORDER BY")
print("✓ LIMIT")
print("✓ DISTINCT")
print("✓ BETWEEN")
print("✓ INNER JOIN")
print("✓ GROUP BY")

print("\nOutput File Created")
print("sql_query_outputs.txt")


print("\nDatabase Connection Closed")

EXECUTING SQL QUERIES

QUERY 1 : SELECT + WHERE
                                                title  rating  price_gbp
0               Sapiens: A Brief History of Humankind       5      54.23
1                                         Set Me Free       5      17.46
2   Scott Pilgrim's Precious Little Life (Scott Pi...       5      52.29
3                           Rip it Up and Start Again       5      35.02
4                          Chase Me (Paris Nights #2)       5      25.27
5                                          Black Dust       5      34.53
6   Worlds Elsewhere: Journeys Around Shakespeareâ...       5      40.30
7   The Four Agreements: A Practical Guide to Pers...       5      17.66
8                                   The Elephant Tree       5      23.82
9                                      Sophie's World       5      15.94
10                        Private Paris (Private #10)       5      47.61
11  #HigherSelfie: Wake Up Your Life. Free Your So...       5      23.11
12 

In [17]:
# ============================================================
# DATA PIPELINE PROJECT
# PART 5 - PANDAS SQL & MERGE COMPARISON
# ============================================================

print("=" * 70)
print("PART 5 - PANDAS SQL & MERGE COMPARISON")
print("=" * 70)

# ------------------------------------------------------------
# Read Books Table into DataFrame
# ------------------------------------------------------------

print("\nReading Books Table...")

books_df = pd.read_sql(

    "SELECT * FROM books",

    connection

)

print("Books Table Loaded Successfully")

print("\nTotal Books :", len(books_df))

# ------------------------------------------------------------
# Read Categories Table into DataFrame
# ------------------------------------------------------------

print("\nReading Categories Table...")

categories_df = pd.read_sql(

    "SELECT * FROM categories",

    connection

)

print("Categories Table Loaded Successfully")

print("\nTotal Categories :", len(categories_df))

# ------------------------------------------------------------
# Read JOIN Result Using SQL
# ------------------------------------------------------------

print("\nExecuting SQL JOIN...")

sql_join_query = """

SELECT

    b.title,

    c.category_name,

    b.rating,

    b.price_gbp,

    b.price_inr,

    b.in_stock

FROM books b

JOIN categories c

ON b.category_id = c.category_id

ORDER BY

    c.category_name,

    b.rating DESC,

    b.title;

"""

sql_join_df = pd.read_sql(

    sql_join_query,

    connection

)

print("SQL JOIN Completed")

# ------------------------------------------------------------
# Perform Same JOIN Using Pandas
# ------------------------------------------------------------

print("\nPerforming Pandas Merge...")

merge_df = pd.merge(

    books_df,

    categories_df,

    on="category_id",

    how="inner"

)

# Keep only required columns

merge_df = merge_df[

    [

        "title",

        "category_name",

        "rating",

        "price_gbp",

        "price_inr",

        "in_stock"

    ]

]

# Sort exactly like SQL

merge_df = merge_df.sort_values(

    by=[

        "category_name",

        "rating",

        "title"

    ],

    ascending=[

        True,

        False,

        True

    ]

).reset_index(drop=True)

# SQL DataFrame reset

sql_join_df = sql_join_df.reset_index(drop=True)

print("Pandas Merge Completed")

# ------------------------------------------------------------
# Compare Both DataFrames
# ------------------------------------------------------------

print("\nComparing SQL JOIN with Pandas Merge...")

comparison = sql_join_df.equals(merge_df)

print("\nAre Both Outputs Equal?")

print(comparison)

# ------------------------------------------------------------
# Display Sample Output
# ------------------------------------------------------------

print("\n")
print("=" * 70)
print("SQL JOIN OUTPUT")
print("=" * 70)

print(sql_join_df.head(10))

print("\n")
print("=" * 70)
print("PANDAS MERGE OUTPUT")
print("=" * 70)

print(merge_df.head(10))

# ------------------------------------------------------------
# Save Outputs
# ------------------------------------------------------------

sql_join_df.to_csv(

    "sql_join_output.csv",

    index=False

)

merge_df.to_csv(

    "pandas_merge_output.csv",

    index=False

)

print("\nCSV Files Saved")

# ------------------------------------------------------------
# Create Comparison Report
# ------------------------------------------------------------

with open(

    "comparison_report.txt",

    "w",

    encoding="utf-8"

) as file:

    file.write("=" * 70 + "\n")

    file.write("SQL JOIN vs PANDAS MERGE COMPARISON\n")

    file.write("=" * 70 + "\n\n")

    file.write("Total Books : ")

    file.write(str(len(books_df)))

    file.write("\n")

    file.write("Total Categories : ")

    file.write(str(len(categories_df)))

    file.write("\n\n")

    file.write("Outputs Match : ")

    file.write(str(comparison))

    file.write("\n")

print("Comparison Report Saved")

# ------------------------------------------------------------
# Display Database Tables
# ------------------------------------------------------------

print("\nDatabase Tables")

print(pd.read_sql(

    "SELECT name FROM sqlite_master WHERE type='table'",

    connection

))

# ------------------------------------------------------------
# Close Database Connection
# ------------------------------------------------------------

connection.close()

print("\nDatabase Connection Closed Successfully")

# ------------------------------------------------------------
# Final Summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PROJECT COMPLETED SUCCESSFULLY")
print("=" * 70)

print("\nFiles Created")

print("✓ raw_books.csv")
print("✓ clean_books.csv")
print("✓ books.db")
print("✓ sql_query_outputs.txt")
print("✓ sql_join_output.csv")
print("✓ pandas_merge_output.csv")
print("✓ comparison_report.txt")

print("\nAssignment Requirements Completed")

print("✓ Web Scraping")
print("✓ Data Cleaning")
print("✓ GBP to INR Conversion")
print("✓ SQLite Database")
print("✓ SQL Queries")
print("✓ Primary Key / Foreign Key")
print("✓ pd.read_sql()")
print("✓ pd.merge()")
print("✓ Output Comparison")

PART 5 - PANDAS SQL & MERGE COMPARISON

Reading Books Table...
Books Table Loaded Successfully

Total Books : 100

Reading Categories Table...
Categories Table Loaded Successfully

Total Categories : 29

Executing SQL JOIN...
SQL JOIN Completed

Performing Pandas Merge...
Pandas Merge Completed

Comparing SQL JOIN with Pandas Merge...

Are Both Outputs Equal?
True


SQL JOIN OUTPUT
                                               title  category_name  rating  \
0  The Mindfulness and Acceptance Workbook for An...  Add a comment       4   
1                                On a Midnight Clear  Add a comment       3   
2                                     The Art Forger  Add a comment       3   
3  Judo: Seven Steps to Black Belt (an Introducto...  Add a comment       2   
4        The Torch Is Passed: A Harding Family Story  Add a comment       1   
5                                     Wall and Piece            Art       4   
6  The Dirty Little Secrets of Getting Your Dream...       Bus